## Laboratorio 2 - Complejidad y búsqueda de hiperparámetros (Grupo G13)

Caso AlpesPlanck: estimación de la temperatura máxima del día siguiente.

En este laboratorio construiremos modelos de regresión polinomial y regularizada (Ridge y Lasso) sobre los datos meteorológicos de AlpesPlanck, evaluando su desempeño y estabilidad mediante validación cruzada, curvas de validación y bootstrapping.

## Preparación de los datos

Reutilizamos el proceso de limpieza y preparación construido en el Laboratorio 1, ya que el enunciado indica que se trabajará con la versión resultante de dicho proceso. A continuación se replican los pasos de limpieza sobre el conjunto de datos original.

## Importación de librerías

Importaremos las librerías necesarias para cargar, explorar, visualizar y preparar el conjunto de datos.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error


ModuleNotFoundError: No module named 'sklearn'

Cargaremos los datos y trabajaremos con una copia para conservar intacto el archivo original.

## Carga y copia de los datos

Cargaremos el conjunto de entrenamiento y crearemos una copia para realizar el análisis sin modificar directamente los datos originales.

In [ ]:
Datos_temperatura = pd.read_csv("./data/Datos Lab 1.csv")
datos = Datos_temperatura.copy()
datos.head()


Usaremos `info()` para revisar la estructura del conjunto de datos, los tipos de las columnas y la cantidad de valores no nulos.

## Inspección inicial

Revisaremos las primeras filas y la estructura del conjunto para confirmar que los datos se cargaron correctamente.

In [ ]:
datos.info()

Consultaremos el diccionario de datos para comprender el significado de cada variable y orientar las decisiones de preparación.

## Diccionario de datos

Consultaremos el diccionario para comprender el significado de las variables y distinguir entre variables numéricas, categóricas y temporales.

In [ ]:
diccionario = pd.read_excel('./data/Diccionario de datos.xlsx')
pd.set_option('display.max_colwidth', None)
diccionario

La columna `fecha` se cargó como texto. La convertiremos a un tipo de fecha para poder extraer y comparar el año, el mes y el día del año.

## Conversión de la fecha

Convertiremos `fecha` al tipo de dato fecha para utilizarla en la validación y en la creación de características temporales.

In [ ]:
datos["fecha"]=pd.to_datetime(datos["fecha"], format="%Y-%m-%d")
datos["fecha"].head()

Después de comprender la estructura de los datos, revisaremos la proporción de valores faltantes en cada columna.

## Valores faltantes

Calcularemos el porcentaje de valores faltantes por columna. Este resultado orientará la estrategia de tratamiento durante la preparación de los datos.

In [ ]:
((datos.isnull().sum()/datos.shape[0])*100).sort_values(ascending=False).round(3)

Revisaremos las estadísticas descriptivas para conocer los rangos, valores centrales y dispersión de las variables numéricas.

## Estadísticas descriptivas

Analizaremos las principales medidas estadísticas de las variables numéricas para identificar su comportamiento y posibles valores atípicos.

In [ ]:
datos.describe()

Después de revisar las estadísticas, verificaremos la existencia de registros duplicados.

## Duplicados completos

Verificaremos si existen filas completamente duplicadas. Estos registros podrían repetir exactamente la misma observación y afectar el análisis.

In [ ]:
print(f"Existen {datos.duplicated(keep = False).sum()} filas duplicadas en los datos.")

## Revisión de fechas repetidas

Comprobaremos cuántas filas comparten la misma fecha. Una fecha repetida no implica necesariamente que la fila completa sea un duplicado.

In [ ]:
print(f"Existen {datos['fecha'].duplicated(keep = False).sum()} filas con fecha duplicada.")

Revisaremos si existen fechas nulas, ya que una fecha faltante impide validar las características temporales del registro.

## Fechas nulas

Contaremos las fechas faltantes porque una fecha nula impide determinar el año, el mes y el día del año asociados al registro.

In [ ]:
datos["fecha"].isnull().sum()

## Eliminación de fechas faltantes

Eliminaremos las filas con fecha faltante, porque no es posible reconstruir de forma confiable las características temporales de esos registros.

In [ ]:
datos= datos.dropna(subset=['fecha'])
datos["fecha"].isnull().sum()

Después de revisar los duplicados completos, inspeccionaremos las filas que tienen fechas repetidas.

## Inspección de fechas repetidas

Mostraremos los registros con fechas repetidas y el número de apariciones de cada fecha. Esto permite distinguir duplicados reales de observaciones diferentes del mismo día.

In [ ]:
dup_table = (
    datos[datos['fecha'].duplicated(keep=False)]
    .copy()
    .assign(Repeticiones=datos.groupby('fecha')['fecha'].transform('size'))
    .sort_values(['Repeticiones', 'fecha'], ascending=[False, True])
)
dup_table

## Ajustes de normalizacion de la Humedad

Aca ajustaremos los valores de la humedad ya que algunos estan entre 0-1 y otros enter 1-100, para esto dejaremos todos entre 0-1 como fraccion y proporcion de porcentaje


In [ ]:
for col in ["humedad_media", "humedad_max", "humedad_min"]:
    datos[col] = datos[col].where(datos[col] <= 1, datos[col] / 100)
    datos[col] = datos[col].clip(upper=1)


## Ajuste de escala de presion

Hay algunos valores de los datos de presión que están en una escala incorrecta (multiplicados por 10, ej. 9784 en vez de 978.4). En vez de descartarlos, corregimos la escala dividiendo entre 10 cuando el valor resultante cae en un rango físicamente válido (900-1100 hPa); solo los valores que sigan siendo inválidos después de este ajuste se marcan como faltantes.

In [ ]:
for col in ["presion_media", "presion_max", "presion_min"]:
    reescalado = datos[col] / 10
    fuera_de_rango = (datos[col] < 900) | (datos[col] > 1100)
    recuperable = fuera_de_rango & reescalado.between(900, 1100)

    print(f"{col}: {fuera_de_rango.sum()} valores fuera de rango, "
          f"{recuperable.sum()} recuperados al corregir la escala (x10).")

    datos[col] = np.where(recuperable, reescalado, datos[col])
    datos[col] = np.where((datos[col] < 900) | (datos[col] > 1100), np.nan, datos[col])

## Depuración de fechas repetidas

Después de inspeccionar los registros repetidos, conservaremos una sola observación por fecha, asumiendo que el conjunto debe contener una fila diaria.



In [ ]:
datos = datos.drop_duplicates(subset=['fecha'], keep='first')

duplicados_restantes = datos['fecha'].duplicated(keep=False).sum()
print(f"Existen {duplicados_restantes} fechas duplicadas después de la limpieza.")

## Controles generales de calidad

Realizaremos verificaciones de tipos de datos, valores faltantes, duplicados y rango temporal después de la limpieza inicial.

In [ ]:
print("Tipos de datos:")
print(datos.dtypes)
print()

faltantes = datos.isna().sum().sort_values(ascending=False)
print("Valores faltantes por columna (solo columnas con al menos 1 faltante):")
print(faltantes[faltantes > 0])
print()

print(f"Filas completamente duplicadas: {datos.duplicated().sum()}")
print(f"Fechas repetidas: {datos['fecha'].duplicated().sum()}")
print(f"Rango de fechas: {datos['fecha'].min().date()} a {datos['fecha'].max().date()}")

## Detección de valores atípicos

Usaremos diagramas de caja para identificar valores extremos en las variables numéricas. Posteriormente con los IQR vamos a quitar los datos anomalos para tener mejor resultado y presicion


In [ ]:
datos.select_dtypes(include="number").boxplot(
    figsize=(18, 8),
    rot=90
)
plt.show()

Aca vemos que hay varios valores outliners que nos dañan la calidad de los datos, para esto los ajustaremos con los IQR para tener intervalos de confianza logicos que nos permitiran distingir y tener mejores datos y resultados, ademas se compararan los resultados en varios boxplot para ver que todo haya funcionado bien

In [ ]:
datos_numericos = (
    datos.select_dtypes(include="number")
    .drop(columns=["temp_max_manana", "registros_del_dia"], errors="ignore")
)

cols_viento = [
    "viento_max", "viento_min", "viento_media", "viento_desv",
    "rafaga_media", "rafaga_max", "rafaga_min", "rafaga_desv"
]

for col in cols_viento:
    datos[col] = pd.to_numeric(datos[col], errors="coerce")
    datos.loc[datos[col] <= -100, col] = np.nan

datos_numericos[cols_viento] = datos[cols_viento]

for column in datos_numericos.columns:
    if column in cols_viento:
        continue

    q1 = datos_numericos[column].quantile(0.25)
    q3 = datos_numericos[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    datos_numericos[column] = datos_numericos[column].apply(
        lambda x: x if lower_bound <= x <= upper_bound else None
    )


datos_numericos.select_dtypes(include="number").boxplot(
    figsize=(18, 8),
    rot=90
)
plt.show()

datos[datos_numericos.columns] = datos_numericos


datos.boxplot(
    column=["presion_media", "presion_max", "presion_min"],
    figsize=(18, 8),
    rot=90
)
plt.show()


Las columnas de viento y ráfaga (`viento_*`, `rafaga_*`) se excluyen deliberadamente del recorte por IQR: son variables con distribuciones naturalmente sesgadas y colas largas (ráfagas fuertes ocasionales son eventos reales, no errores), por lo que aplicar el mismo criterio que a las demás variables numéricas recortaría información climática válida. En su lugar, para estas columnas solo se tratan como faltantes los valores físicamente imposibles (`<= -100`, ver más arriba).

In [ ]:
datos.boxplot(
    column=["humedad_media", "humedad_max", "humedad_min","presion_desv"],
    figsize=(18, 8),
    rot=90
)

plt.show()

In [ ]:
datos.boxplot(
    column=["viento_media", "viento_max", "viento_min","viento_desv"],
    figsize=(18, 8),
    rot=90
)

plt.show()

In [ ]:
datos.boxplot(
    column=["rafaga_media", "rafaga_max", "rafaga_min","rafaga_desv"],
    figsize=(18, 8),
    rot=90
)

plt.show()

In [ ]:
datos.select_dtypes(include="object").nunique()
for columna in datos.select_dtypes(include="object"):
    print(f"\n{columna}:")
    print(datos[columna].value_counts(dropna=False))

Al revisar las variables temporales y categóricas, identificamos posibles inconsistencias en los años, meses, días, estaciones y sectores del viento. Las corregiremos usando la fecha y las variables relacionadas.

In [ ]:
datos["anio"] = datos["fecha"].dt.year
datos["dia_del_anio"] = datos["fecha"].dt.dayofyear

In [ ]:
datos["mes"].unique()

In [ ]:
meses_es = {
    "January": "enero", "February": "febrero", "March": "marzo", "April": "abril",
    "May": "mayo", "June": "junio", "July": "julio", "August": "agosto",
    "September": "septiembre", "October": "octubre", "November": "noviembre", "December": "diciembre"
}
datos["mes"] = datos["fecha"].dt.month_name().map(meses_es)
datos["mes"].unique()


Aca se va a poner los dias como variables ciclicas con funciones de seno y coseno ya que asi se nota mas la correlacion que esta informacion tiene con la variable objetivo que simplemente el dato a secas como un numero de dia, esto ya que al volverlo ciclico se ve las la correlacion.



In [ ]:
datos["dia_del_anio"] = datos["fecha"].dt.dayofyear

datos["dia_sin"] = np.sin(2 * np.pi * datos["dia_del_anio"] / 365.25)
datos["dia_cos"] = np.cos(2 * np.pi * datos["dia_del_anio"] / 365.25)

## Tratamiento de valores faltantes

Revisaremos nuevamente los valores faltantes y definiremos una estrategia según el tipo y el significado de cada variable.

In [ ]:
datos.isna().sum().sort_values(ascending=False)

Como `temp_max_manana` es la variable objetivo, eliminaremos los registros que no tienen esta etiqueta, ya que no pueden utilizarse para entrenar el modelo.

In [ ]:
datos=datos.dropna(subset=["temp_max_manana"])
datos["temp_max_manana"].isnull().sum()

Para las variables numéricas predictoras con valores faltantes, utilizaremos la mediana. Esta medida es menos sensible a los valores atípicos que el promedio.

In [ ]:
columnas_numericas = datos.select_dtypes(include="number").columns.drop(
    ["temp_max_manana", "direccion_viento"],
    errors="ignore"
)

medianas_entrenamiento = datos[columnas_numericas].median()
datos[columnas_numericas] = datos[columnas_numericas].fillna(medianas_entrenamiento)

Calcularemos los valores faltantes de `direccion_viento` a partir de las componentes `viento_este` y `viento_norte`, que ya fueron completadas.

In [ ]:
direccion_calculada = (
    np.degrees(
        np.arctan2(datos["viento_este"], datos["viento_norte"])
    ) % 360
)

datos["direccion_viento"] = datos["direccion_viento"].fillna(
    direccion_calculada
)

Revisaremos los valores de `estacion_anio` y completaremos los faltantes usando la fecha como fuente de información.

In [ ]:
datos["estacion_anio"].unique()

In [ ]:
datos["estacion_anio"] = datos["fecha"].dt.month.map({
    1: "invierno",
    2: "invierno",
    3: "primavera",
    4: "primavera",
    5: "primavera",
    6: "verano",
    7: "verano",
    8: "verano",
    9: "otoño",
    10: "otoño",
    11: "otoño", 
    12: "invierno"
})

datos["estacion_anio"].unique()

In [ ]:
estacion_calculada = datos["fecha"].dt.month.map({
    1: "invierno",
    2: "invierno",
    3: "primavera",
    4: "primavera",
    5: "primavera",
    6: "verano",
    7: "verano",
    8: "verano",
    9: "otoño",
    10: "otoño",
    11: "otoño",
    12: "invierno"
})

datos["estacion_anio"] = datos["estacion_anio"].fillna(estacion_calculada)

Completaremos los valores faltantes de `sector_viento` usando `direccion_viento` como referencia.

In [ ]:
sectores = np.array(["N", "NE", "E", "SE", "S", "SO", "O", "NO"])

indice_sector = (
    ((datos["direccion_viento"] + 22.5) // 45) % 8
).astype(int)

sector_calculado = pd.Series(
    sectores[indice_sector],
    index=datos.index
)

datos["sector_viento"] = datos["sector_viento"].fillna(
    sector_calculado
)

## Verificación de valores faltantes

Confirmaremos que no queden valores nulos después de aplicar las estrategias de limpieza.

In [ ]:
datos.isna().sum().sort_values(ascending=False)

## Verificación de sectores del viento

Revisaremos las categorías resultantes para confirmar que los sectores estén unificados.

In [ ]:
datos["sector_viento"].unique()

In [ ]:
datos["sector_viento"] = (
    datos["sector_viento"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "NORTE": "N",
        "NORTH": "N",
        "SUR": "S",
        "SOUTH": "S",
        "ESTE": "E",
        "EAST": "E",
        "OESTE": "O",
        "WEST": "O",
        "NORESTE": "NE",
        "NORTHEAST": "NE",
        "NOROESTE": "NO",
        "NORTHWEST": "NO",
        "SURESTE": "SE",
        "SOUTHEAST": "SE",
        "SUROESTE": "SO",
        "SOUTHWEST": "SO",
        "SW": "SO",
        "NW": "NO",
        "W": "O"
    })
)

datos["sector_viento"].unique()

## Verificación de consistencia

Comprobaremos que las variables temporales, las direcciones del viento y las categorías resultantes tengan valores válidos y coherentes. Adicionalmente, eliminamos los registros con `temp_max_manana` por encima de 35°C en meses fríos, por considerarse climatológicamente implausibles para esta ubicación. Esta es una decisión de limpieza basada en la variable objetivo, por lo que se reporta explícitamente cuántas filas afecta para poder discutirla como posible fuente de sesgo en el análisis de resultados.

In [ ]:

assert (datos["anio"] == datos["fecha"].dt.year).all()
assert (datos["dia_del_anio"] == datos["fecha"].dt.dayofyear).all()

assert datos["dia_del_anio"].between(1, 366).all()
assert datos["direccion_viento"].between(0, 360).all()

assert datos.isna().sum().sum() == 0

assert datos.duplicated().sum() == 0

assert datos["fecha"].notna().all()

assert set(datos["estacion_anio"].unique()) <= {
    "invierno", "primavera", "verano", "otoño"
}

assert set(datos["sector_viento"].unique()) <= {
    "N", "NE", "E", "SE", "S", "SO", "O", "NO"
}

filas_antes_filtro = datos.shape[0]
filtro_temp_imposible = (
    datos["fecha"].dt.month.isin([1, 2, 3, 4, 10, 11, 12])
    & (datos["temp_max_manana"] > 35)
)

print(
    f"Se identificaron {filtro_temp_imposible.sum()} registros con temp_max_manana > 35°C "
    "en meses de otoño, invierno o inicio/fin de primavera, considerados climatológicamente "
    "inconsistentes para la ubicación de AlpesPlanck y por tanto tratados como error de registro."
)

datos = datos[~filtro_temp_imposible]

print(
    f"Filas antes del filtro: {filas_antes_filtro} | filas después: {datos.shape[0]} "
    f"(se eliminaron {filas_antes_filtro - datos.shape[0]} filas, "
    f"{100 * (filas_antes_filtro - datos.shape[0]) / filas_antes_filtro:.2f}% del total)."
)

**Nota metodológica:** las medianas de imputación y los límites IQR usados en esta sección se calcularon una única vez sobre todo el conjunto de entrenamiento, antes de cualquier partición. Esto es razonable para esta etapa de preparación (el enunciado indica que se parte de una versión ya limpia de los datos), pero implica una fuga leve de información hacia los folds de validación cruzada que se usarán más adelante en la búsqueda de hiperparámetros: dichos folds ya están, en parte, reflejados en las medianas y los límites de recorte calculados aquí. Se deja explícito como limitación a considerar en el análisis crítico.